# TabICL fold0 baseline — exp_070 (tabular foundation model, ICL)

S6E5 1위·8위 핵심 멤버. features=base(raw, S6E5 정석), offload_mode=auto+batch_size=2(440k 메모리).
**추론 단일 GPU**(T4x1 충분, multi-GPU는 fine-tuning 전용). cell3.5 소규모 테스트로 메모리 fast-fail.
게이트: fold0 corr<0.97(분기) + 개별~0.95.

In [ ]:
# 1) input 자동탐색 (마운트 비표준: /kaggle/input/{datasets,competitions}/...) — torch import 前
import sys, os, glob, subprocess
from pathlib import Path
print('/kaggle/input:', os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'NONE')
c = glob.glob('/kaggle/input/**/src/config.py', recursive=True); assert c, 'src/config.py 못 찾음'
SRC_ROOT = str(Path(c[0]).parents[1]); print('SRC_ROOT:', SRC_ROOT)
cc = glob.glob('/kaggle/input/**/playground-series-s6e5', recursive=True); assert cc, '대회 폴더 못 찾음'
COMP = Path(cc[0]); print('COMP:', COMP)
ac = glob.glob('/kaggle/input/**/f1_strategy_dataset*.csv', recursive=True); assert ac, '증강 csv 못 찾음'
AUG = Path(ac[0]); print('AUG:', AUG)

In [ ]:
# 2) tabicl 설치 + GPU 확인 (torch는 tabicl 의존으로 함께)
import sys, subprocess
print('GPU:', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
pip('tabicl','hydra-core','python-dotenv')
import tabicl, torch
print('tabicl', getattr(tabicl,'__version__','?'), '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())

In [ ]:
# 3) src import
sys.path.insert(0, SRC_ROOT)
from src import config
from src.train_tabicl import run
print('import OK:', config.__file__)

In [ ]:
# 3.5) 소규모 fast-fail (10k행 TabICL fit/predict — API·GPU 메모리 검증)
import pandas as pd, numpy as np, time
from tabicl import TabICLClassifier
_tr = pd.read_csv(COMP/'train.csv').sample(10000, random_state=42)
_y = _tr['PitNextLap']; _X = _tr.drop(columns=['id','PitNextLap'])
for c in _X.select_dtypes('object').columns: _X[c]=_X[c].astype('category').cat.codes
t0=time.time(); m=TabICLClassifier(device='cuda', n_estimators=8, batch_size=2, offload_mode='auto', random_state=42)
m.fit(_X,_y); p=m.predict_proba(_X.head(2000))[:,1]
print(f'TabICL 10k fit+pred OK ✓ {time.time()-t0:.0f}s | pred {p.mean():.3f} | GPU mem 한계 없으면 full 진행')

In [ ]:
# 4) 경로 override
config.TRAIN_PATH = COMP / 'train.csv'
config.TEST_PATH = COMP / 'test.csv'
config.SAMPLE_SUBMISSION_PATH = COMP / 'sample_submission.csv'
config.SOURCE_AUG_PATH = AUG
out = Path('/kaggle/working')
config.OOF_DIR = out / 'oof'; config.SUBMISSION_DIR = out / 'submissions'; config.LOG_DIR = out / 'logs'
import pandas as pd
_a = pd.read_csv(config.SOURCE_AUG_PATH); print('AUG shape:', _a.shape)
assert len(_a) == 101371, f'증강 행수 불일치: {len(_a)}'
assert config.TRAIN_PATH.exists(), f'train.csv 없음: {config.TRAIN_PATH}'

In [ ]:
# 5) cfg + run — TabICL baseline(raw=base), fold0
from omegaconf import OmegaConf
import time
CONF = Path(SRC_ROOT) / 'conf'
cfg = OmegaConf.create({
    'exp_id': 'exp_070_tabicl_baseline_fold0',
    'notes': 'TabICL baseline fold0: base(raw) features, offload auto, NN ICL 분기 테스트',
    'use_wandb': False, 'max_folds': 1,
    'model': OmegaConf.load(CONF / 'model' / 'tabicl.yaml'),
    'features': OmegaConf.load(CONF / 'features' / 'base.yaml'),
    'augment': {'enabled': False, 'weight': 1.0},
})
print(OmegaConf.to_yaml(cfg))
t0=time.time(); result=run(cfg); print(result, f'{time.time()-t0:.0f}s')
print('fold0 AUC =', result.get('fold_scores',[None])[0])